In [1]:
import torch
import os
import gc
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import CLIPTextModel, CLIPTokenizer
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler, StableDiffusionPipeline
from diffusers.optimization import get_scheduler
from peft import LoraConfig, get_peft_model, PeftModel
import bitsandbytes as bnb
from itertools import cycle
# --- KONFIGURACJA ---
MODEL_ID = "runwayml/stable-diffusion-v1-5"
INSTANCE_DIR = "./FIXED21"
OUTPUT_DIR = "./lora_weights_1"
PROMPT = "a photo of sks_mug"
RESOLUTION = 512
BATCH_SIZE = 1
STEPS = 1000

In [ ]:
# --- DATASET ---
class SimpleDataset(Dataset):
    def __init__(self, folder, tokenizer, size=512):
        self.images = [os.path.join(folder, f) for f in os.listdir(folder)]
        self.tokenizer = tokenizer
        self.transforms = transforms.Compose([
            transforms.Resize(size, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.CenterCrop(size),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])

    def __len__(self): return len(self.images)

    def __getitem__(self, i):
        image = Image.open(self.images[i]).convert("RGB")
        pixel_values = self.transforms(image)

        # Tokenizacja bez zbędnych wymiarów batcha
        tokenized = self.tokenizer(
            PROMPT,
            padding="max_length",
            truncation=True,
            max_length=self.tokenizer.model_max_length,
            return_tensors="pt"
        ).input_ids[0]

        return {
            "pixel_values": pixel_values,
            "input_ids": tokenized
        }

# --- ŁADOWANIE MODELI ---
# VAE na CPU (float32) - tylko do jednorazowego kodowania
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae").to("cpu", dtype=torch.float32)

# Text Encoder na GPU (float16) - tylko do jednorazowego kodowania
tokenizer = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder", torch_dtype=torch.float16).to("cuda")

# UNet na GPU (float32 dla stabilności treningu, LoRA i tak mało waży)
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet").to("cuda")
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

# --- PRZYGOTOWANIE LORA ---
lora_config = LoraConfig(
    r=16, target_modules=["to_q", "to_k", "to_v", "to_out.0"],
    lora_dropout=0.05, bias="none"
)
unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()

optimizer = bnb.optim.AdamW8bit(unet.parameters(), lr=1e-4)
lr_scheduler = get_scheduler("cosine", optimizer=optimizer, num_training_steps=STEPS, num_warmup_steps=50)

# Dataset i DataLoader
dataset = SimpleDataset(INSTANCE_DIR, tokenizer, RESOLUTION)
train_dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# --- FAZA CACHE'OWANIA (Pre-compute) ---
print(">>> Cache'owanie Latentów i Embeddingów... (Cierpliwości)")
cached_data = []

# Blokujemy gradienty dla fazy cache
vae.requires_grad_(False)
text_encoder.requires_grad_(False)

for batch in train_dataloader:
    # 1. Obraz -> Latent
    pixel_values = batch["pixel_values"].to("cpu", dtype=torch.float32) # VAE woli float32 na CPU
    with torch.no_grad():
        latents = vae.encode(pixel_values).latent_dist.sample() * 0.18215
    # Przenosimy latent do VRAM jako fp16
    latents = latents.to("cuda", dtype=torch.float16)

    # 2. Tekst -> Embedding
    input_ids = batch["input_ids"].to("cuda")
    with torch.no_grad():
        encoder_hidden_states = text_encoder(input_ids)[0].to(dtype=torch.float16)

    # Zapisujemy parę (Latent, Embedding) do listy
    cached_data.append((latents, encoder_hidden_states))

print(f">>> Zcache'owano {len(cached_data)} batchy.")

# --- MEMORY CLEANUP ---
# Usuwamy wszystko co zbędne. Zostaje tylko UNet i Cache.
del vae
del text_encoder
del tokenizer
del dataset
del train_dataloader
gc.collect()
torch.cuda.empty_cache()
print(">>> Zwolniono VRAM. TextEncoder i VAE usunięte.")

# --- PĘTLA TRENINGOWA ---
unet.enable_gradient_checkpointing()
unet.train()

# Iterator cykliczny po zcache'owanych danych
train_iter = cycle(cached_data)

print(">>> Start Treningu...")

for step in range(STEPS):
    # Pobieramy gotową parę z VRAM
    latents, encoder_hidden_states = next(train_iter)

    with torch.amp.autocast('cuda'):
        # Generujemy szum
        noise = torch.randn_like(latents)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],), device="cuda").long()

        # Dodajemy szum do latenta
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        # Predykcja (UNet ma już gotowe embeddingi!)
        noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample

        # Obliczamy loss (rzutujemy na float32 dla precyzji)
        loss = torch.nn.functional.mse_loss(noise_pred.float(), noise.float())

    # Backprop
    loss.backward()
    optimizer.step()
    lr_scheduler.step()
    optimizer.zero_grad()

    if step % 50 == 0:
        print(f"Step {step}, Loss: {loss.item():.4f}")

# Zapis wyników
print(">>> Zapisywanie wag...")
unet.save_pretrained(OUTPUT_DIR)
print(">>> Gotowe.")

In [2]:
import gc
try:
    del unet
    del vae
    del text_encoder
    del optimizer
    del train_dataloader
except NameError:
    pass

gc.collect()

torch.cuda.empty_cache()
pipe = StableDiffusionPipeline.from_pretrained(MODEL_ID).to("cuda")
import random
pipe.unet = PeftModel.from_pretrained(pipe.unet, OUTPUT_DIR)
generator = torch.Generator("cuda").manual_seed(random.randint(0, 1000))


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

In [9]:
prompt = "a photo of sks_mug, on beach"
negative_prompt = "blurry, distorted, low quality, deformed"

images = pipe(
    prompt,
    # negative_prompt=negative_prompt,
    num_inference_steps=40,
    guidance_scale=9,
    generator=generator
).images[0]
images.save(f"Cactus.png")
print("All images saved")

  0%|          | 0/40 [00:00<?, ?it/s]

All images saved
